In [ ]:
#!/usr/bin/env python
# coding: utf-8

# Install necessary libraries
get_ipython().system('pip install -U transformers datasets accelerate peft trl bitsandbytes')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Foun

In [ ]:
# Log in to Hugging Face
get_ipython().system('huggingface-cli login')


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `mariamattiaa` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenti

In [ ]:
# Define paths and model configurations
base_model = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
import torch

# Check if GPU is available
print("CUDA Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


CUDA Available: False
Device Name: No GPU detected


In [ ]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Load the base model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))


In [ ]:
# Function to generate prompt with Gardiner codes and meanings
def generate_prompt_with_data(dataframe):
    base_prompt = (
        "You are an expert in translating Gardiner codes into meaningful and accurate English sentences. "
        "Your task involves the following steps: \n"
        "1. For a single Gardiner code, provide its meaning clearly and concisely without unnecessary details.\n"
        "2. For multiple Gardiner codes, explain each code individually and then combine their meanings into a coherent sentence.\n"
        "3. If you encounter an unknown Gardiner code, respond with 'I do not know' and avoid fabricating answers.\n"
        "Guidelines: \n"
        "- Keep responses accurate and professional.\n"
        "- Only use information provided in the query.\n"
        "- Avoid referencing unrelated codes unless explicitly requested.\n"
        "Available Gardiner codes and their meanings: \n"
    )
        codes_meanings = "\n".join(
        f"{row['gardiner_code']}: {row['english_translation']}"
        for _, row in dataframe.iterrows()
    )

    return base_prompt + codes_meanings

In [ ]:
# Load data and generate the full prompt
data_path = '/mnt/data/cleaned_dataset2.csv'
data = pd.read_csv(data_path, encoding='latin1')
explanation_prompt = generate_prompt_with_data(data)

In [ ]:
# Generate text using the model
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.5  # Lower temperature for more deterministic output
)

In [ ]:
# Single Gardiner code query
query = f"{explanation_prompt}\nWhat does Gardiner code 'A5' mean?"
response = llm_pipeline(query)
print(f"Generated Response for Single Code: {response[0]['generated_text']}")


In [ ]:
# Multiple Gardiner codes query
query = f"{explanation_prompt}\nWhat do Gardiner codes 'A5', 'A10', and 'A3' mean?"
response = llm_pipeline(query)
print(f"Generated Response for Multiple Codes: {response[0]['generated_text']}")

In [ ]:
# Evaluation Metrics
def evaluate_model(predictions, references):
    accuracy = accuracy_score(references, predictions)
    precision = precision_score(references, predictions, average='weighted')
    recall = recall_score(references, predictions, average='weighted')
    f1 = f1_score(references, predictions, average='weighted')

    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")


In [ ]:
# Example Evaluation
# Replace with actual predictions and references for real evaluation
sample_predictions = ["man", "hide", "sit"]
sample_references = ["man", "hide", "sit"]
evaluate_model(sample_predictions, sample_references)